# Análise exploratória — Vítimas de auto de resistência no Rio de Janeiro (2003–2022)

**Autor:** Matheus Gomes da Costa · **Curso:** Sistemas de Informação, 6º período · **Disciplina:** Linguagem de Programação (Análise e Visualização de Dados com Python)
**Fonte:** Data.Rio, Tabela 2300 (ISP-RJ/IPP) · **Licenças:** código MIT, dados e texto CC-BY 4.0

Notebook de exploração e validação. A lógica reutilizável está em `src/etl.py`; aqui ela é executada passo a passo e os resultados são conferidos.

**Perguntas de pesquisa**
- **RQ1** A série segue tendência linear ou se organiza em patamares?
- **RQ2** A queda da taxa por 100 mil habitantes vem do número de vítimas ou do crescimento populacional?
- **RQ3** Quão volátil é a série e como as maiores variações se situam frente a marcos do período?

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # permite `import src`

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

from src import etl
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Dado bruto
A planilha tem título, notas e células vazias ao redor da grade.

In [2]:
raw = etl.load_raw()
print(raw.shape)
raw.iloc[[2, 6, 7, 10, 11, 29, 32]].iloc[:, :7]

(39, 7)


,0,1,2,3,4,5,6
2,"Tabela 2300 - Vítimas de auto de resistência, ...",NaN,NaN,NaN,NaN,NaN,NaN
6,Ano \n1º quadrimestre,Vítimas de auto de resistência ocorrido no MRJ1,NaN,NaN,NaN,NaN,NaN
7,NaN,Total - Jan-Abr,Diferença Absoluta3,Diferença %3,População2,Taxa por 100 mil habitantes3,Média mensal3
10,2003,291,..,..,"5,865,737.42",4.96,72.75
11,2004,203,-88,-30.24,"6,027,371.27",3.37,50.75
29,2022,123,-98,-44.34,"6,782,154.88",1.81,30.75
32,Fonte: Instituto de Segurança Pública. Dispon...,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Limpeza e validação
Isolamos as linhas de ano, convertemos `..` em `NaN` e conferimos as métricas derivadas da fonte.

In [3]:
df, log = etl.clean(raw)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.dtypes

{
  "linhas_brutas": 39,
  "linhas_de_dados": 20,
  "linhas_descartadas_titulo_notas_vazias": 19,
  "valores_'..'_convertidos_para_NaN": {
    "diferenca_absoluta": 1,
    "diferenca_pct": 1
  },
  "nulos_por_coluna": {
    "diferenca_absoluta": 1,
    "diferenca_pct": 1
  },
  "anos": [
    2003,
    2022
  ]
}


ano                     int64
total_jan_abr           Int64
diferenca_absoluta      Int64
diferenca_pct         float64
populacao             float64
taxa_100k             float64
media_mensal          float64
dtype: object

In [4]:
validacao = etl.validate(df)
pd.DataFrame(validacao).T

,erro_abs_max,consistente
diferenca_absoluta,0.00,True
diferenca_pct,0.00,True
taxa_100k,0.00,True
media_mensal,0.00,True


## 3. Métricas derivadas e estatística descritiva

In [5]:
df, segs = etl.add_metrics(df)
resumo = etl.summary_stats(df, segs)
df[["ano", "total_jan_abr", "diferenca_pct", "taxa_100k", "media_movel_3a", "indice_2003", "z_score", "outlier_iqr", "periodo"]]

,ano,total_jan_abr,diferenca_pct,taxa_100k,media_movel_3a,indice_2003,z_score,outlier_iqr,periodo
0,2003,291,NaN,4.96,NaN,100.00,1.46,False,P1 (2003–2010)
1,2004,203,-30.24,3.37,228.00,69.80,0.24,False,P1 (2003–2010)
2,2005,190,-6.40,3.13,199.33,65.30,0.06,False,P1 (2003–2010)
3,2006,205,7.89,3.35,227.67,70.40,0.27,False,P1 (2003–2010)
4,2007,288,40.49,4.67,274.67,99.00,1.42,False,P1 (2003–2010)
5,2008,331,14.93,5.33,282.00,113.70,2.01,False,P1 (2003–2010)
6,2009,227,-31.42,3.63,256.67,78.00,0.57,False,P1 (2003–2010)
7,2010,212,-6.61,3.36,182.00,72.90,0.36,False,P1 (2003–2010)
8,2011,107,-49.53,1.68,137.67,36.80,-1.10,False,P2 (2011–2016)
9,2012,94,-12.15,1.47,92.67,32.30,-1.28,False,P2 (2011–2016)


In [6]:
df["total_jan_abr"].astype(float).describe()

count    20.00
mean    185.95
std      72.02
min      77.00
25%     120.00
50%     196.50
75%     227.00
max     331.00
Name: total_jan_abr, dtype: float64

### Outliers
Pelo critério do IQR (Tukey) e pelo *z-score*:

In [7]:
print("Outliers (IQR):", resumo["outliers_iqr"] or "nenhum")
df.loc[df.z_score.abs().nlargest(3).index, ["ano", "total_jan_abr", "z_score"]]

Outliers (IQR): nenhum


,ano,total_jan_abr,z_score
5,2008,331,2.01
10,2013,77,-1.51
0,2003,291,1.46


## 4. RQ1 — Tendência linear ou patamares?
Ajuste linear + teste de Kendall, depois segmentação em 3 patamares por mínimos quadrados.

In [8]:
x, y = df["ano"], df["total_jan_abr"].astype(float)
lin = stats.linregress(x, y)
tau, p_tau = stats.kendalltau(x, y)
print(f"inclinação = {lin.slope:.2f}/ano · R² = {lin.rvalue**2:.3f} · p = {lin.pvalue:.3f}")
print(f"Kendall τ = {tau:.3f} · p = {p_tau:.3f}")

inclinação = -4.39/ano · R² = 0.130 · p = 0.118
Kendall τ = -0.159 · p = 0.330


In [9]:
pd.DataFrame(resumo["periodos"])

,rotulo,ano_ini,ano_fim,n,media,desvio,min,max,taxa_media
0,2003–2010,2003,2010,8,243.40,52.30,190,331,3.97
1,2011–2016,2011,2016,6,101.80,18.30,77,130,1.58
2,2017–2022,2017,2022,6,193.50,41.30,123,230,2.89


In [10]:
fig = go.Figure()
fig.add_scatter(x=df.ano, y=df.total_jan_abr, mode="lines+markers", name="Vítimas (jan–abr)", line_color="#1c2a3a")
fig.add_scatter(x=df.ano, y=df.media_movel_3a, mode="lines", name="Média móvel 3 anos", line=dict(color="#b9ad94", width=4))
for p, cor in zip(resumo["periodos"], ["#1f3a5f", "#2a7f78", "#c2501c"]):
    fig.add_shape(type="line", x0=p["ano_ini"]-.5, x1=p["ano_fim"]+.5, y0=p["media"], y1=p["media"], line=dict(color=cor, dash="dot"))
fig.update_layout(template="simple_white", title="Vítimas por ano e patamares", yaxis_title="Vítimas no 1º quadrimestre", height=420)
fig.show()

## 5. RQ2 — Numerador ou denominador?
`ln(taxa_f / taxa_i) = ln(vítimas_f / vítimas_i) − ln(pop_f / pop_i)`: separa o efeito do número de vítimas do efeito demográfico.

In [11]:
d = resumo["decomposicao_taxa"]
print(d)
print(f"A redução de vítimas explica {d['parcela_numerador_pct']:.1f}% da queda da taxa; população cresceu {resumo['populacao_var_pct']:.1f}%.")
rho, p = stats.spearmanr(df.populacao, df.total_jan_abr)
print(f"Spearman(população, vítimas) = {rho:.3f} (p = {p:.3f})")

{'ln_taxa': -1.006, 'ln_total': -0.861, 'ln_populacao': -0.145, 'parcela_numerador_pct': 85.6}
A redução de vítimas explica 85.6% da queda da taxa; população cresceu 15.6%.
Spearman(população, vítimas) = -0.265 (p = 0.259)


In [12]:
fig = go.Figure()
fig.add_scatter(x=df.ano, y=df.indice_2003, name="Vítimas", line_color="#1c2a3a")
fig.add_scatter(x=df.ano, y=df.taxa_100k / df.taxa_100k.iloc[0] * 100, name="Taxa por 100 mil", line_color="#c2501c")
fig.add_scatter(x=df.ano, y=df.populacao / df.populacao.iloc[0] * 100, name="População", line_color="#b9ad94")
fig.update_layout(template="simple_white", title="Índice (2003 = 100)", height=400)
fig.show()

## 6. RQ3 — Volatilidade interanual

In [13]:
v = df.dropna(subset=["diferenca_pct"])
print("Maiores altas:");  display(v.nlargest(3, "diferenca_pct")[["ano", "total_jan_abr", "diferenca_absoluta", "diferenca_pct"]])
print("Maiores quedas:"); display(v.nsmallest(3, "diferenca_pct")[["ano", "total_jan_abr", "diferenca_absoluta", "diferenca_pct"]])
print(f"{resumo['anos_com_alta']} altas · {resumo['anos_com_queda']} quedas · {len(v) - resumo['anos_com_alta'] - resumo['anos_com_queda']} repetição")

Maiores altas:


,ano,total_jan_abr,diferenca_absoluta,diferenca_pct
14,2017,180,69,62.16
12,2015,130,38,41.30
4,2007,288,83,40.49


Maiores quedas:


,ano,total_jan_abr,diferenca_absoluta,diferenca_pct
8,2011,107,-105,-49.53
19,2022,123,-98,-44.34
6,2009,227,-104,-31.42


7 altas · 11 quedas · 1 repetição


In [14]:
cores = ["#c2501c" if x > 0 else "#2a7f78" for x in v.diferenca_pct]
fig = go.Figure(go.Bar(x=v.ano, y=v.diferenca_pct, marker_color=cores))
fig.update_layout(template="simple_white", title="Variação % sobre o ano anterior", yaxis_ticksuffix="%", height=380)
fig.show()

## 7. Conclusões e limitações
- Sem tendência linear detectável (R² = 0,13); a série se organiza em três patamares (≈243 → ≈102 → ≈194).
- ≈86% da queda da taxa vem do número de vítimas, não do crescimento populacional.
- Análise **descritiva**: 20 observações, só o 1º quadrimestre, classificação policial provisória e ponto de 2022 possivelmente revisado. Marcos institucionais (UPP 2008, intervenção federal 2018, ADPF 635 em 2020) são contexto, não causas testadas.